In [0]:
# Bronze -> Silver for `portfolio_company` (internal CSV, 30 rows).
# benchmark_ticker is OPTIONAL (null for 6/30 rows, expected per the
# transformation doc - not every company has a public benchmark). Run AFTER 01_silver_fund.


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from datetime import date

SOURCE_NAME = "portfolio_company"
REQUIRED_COLS = ["company_id", "company_name", "fund_id", "industry"]   
KEY_COLS = ["company_id"]
COMPARE_COLS = ["company_name", "fund_id", "industry", "benchmark_ticker"]
business_date_str = date.today().isoformat()

In [0]:
bronze_df = read_bronze(spark, SOURCE_NAME)
print(f"Bronze row count: {bronze_df.count()}")

Bronze row count: 150


In [0]:
typed_df = (
    bronze_df
    .withColumn("company_id", F.trim(F.col("company_id")))
    .withColumn("company_name", F.trim(F.col("company_name")))
    .withColumn("fund_id", F.trim(F.col("fund_id")))
    .withColumn("industry", F.trim(F.col("industry")))
    .withColumn("benchmark_ticker", F.trim(F.col("benchmark_ticker")))
    .withColumn("has_benchmark", F.col("benchmark_ticker").isNotNull()) 
)

In [0]:
clean_df, null_rejects_df = split_on_required_nulls(typed_df, REQUIRED_COLS)
null_reject_count = null_rejects_df.count()
if null_reject_count > 0:
    write_quarantine(null_rejects_df, SOURCE_NAME)

### Referential integrity - fund_id must resolve

In [0]:
fund_silver = spark.table(silver_table("fund"))
valid_fk_df, invalid_fk_df = check_foreign_key(clean_df, "fund_id", fund_silver, "fund_id")
invalid_fk_count = invalid_fk_df.count()
if invalid_fk_count > 0:
    write_quarantine(invalid_fk_df, SOURCE_NAME)

In [0]:
deduped_df, duplicates_df, breaks_df = split_duplicates(valid_fk_df, KEY_COLS, COMPARE_COLS)
dup_count = duplicates_df.count()
break_count = breaks_df.count()
if dup_count > 0:
    write_quarantine(duplicates_df, SOURCE_NAME)
if break_count > 0:
    write_quarantine(breaks_df.withColumn("reason_code", F.lit("COMPANY_ATTRIBUTE_BREAK")), SOURCE_NAME)

In [0]:
write_silver(deduped_df, SOURCE_NAME)
print(f"Silver row count: {deduped_df.count()}")
print(f"Companies with no benchmark: {deduped_df.filter(~F.col('has_benchmark')).count()} (expected, documented per Section 3.7)")

Silver row count: 30
Companies with no benchmark: 6 (expected, documented per Section 3.7)


In [0]:
log_dq(spark, SOURCE_NAME, business_date_str, "null_required_field", bronze_df.count(), null_reject_count, "NULL_REQUIRED_FIELD")
log_dq(spark, SOURCE_NAME, business_date_str, "unknown_reference", clean_df.count(), invalid_fk_count, "UNKNOWN_REFERENCE")
log_dq(spark, SOURCE_NAME, business_date_str, "duplicate_record", valid_fk_df.count(), dup_count, "DUPLICATE_RECORD")
log_dq(spark, SOURCE_NAME, business_date_str, "attribute_break", valid_fk_df.count(), break_count, "COMPANY_ATTRIBUTE_BREAK")

/home/spark-11f53a55-1081-4354-9c7f-5a/.ipykernel/71/command-5696143635338715-3986853518:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
quarantined_count = null_reject_count + invalid_fk_count + dup_count
assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")

OK: bronze=150 = silver=30 + quarantined=120
